# 01B — Split filtered variables into reusable groups

This notebook loads the scenario-specific filtered variable table from 01A and separates variables into useful groups: patient targets, tunable coefficients, output targets, constants, switches, and diagnostic variables. These groups will be saved and reused in the patient-specific setup notebook.

### Imports and paths

In [14]:
import json
import pandas as pd
from pathlib import Path

data_dir = Path("../01_Data")
processed_dir = data_dir / "processed_pediatric_noVAD_linear"

filtered_path = processed_dir / "classified_variables_pediatric_dcm_noVAD_linear_filtered.csv"

filtered = pd.read_csv(filtered_path)

filtered.head()

,variable,category,tunable,explanation,equation,flagged,simulink_usage,simulink_users,scenario_relevance,reason_filtering,keep_for_current_pipeline
0,weight,fixed_patient_target,No,Patient body weight,NaN,NaN,not_directly_used,NaN,active_or_unknown,NaN,True
1,BSA,fixed_patient_target,No,Body surface area: estimated total surface are...,NaN,NaN,not_directly_used,NaN,active_or_unknown,NaN,True
2,hr,fixed_patient_target,No,Heart rate,NaN,NaN,directly_used,MyComplexModel_V13R2023a/Constant10 | MyComple...,active_or_unknown,NaN,True
3,SAP,fixed_patient_target,No,Systolic arterial pressure; origin: Gupta,NaN,NaN,not_directly_used,NaN,active_or_unknown,NaN,True
4,DAP,fixed_patient_target,No,Diastolic arterial pressure; origin: Gupta,NaN,NaN,not_directly_used,NaN,active_or_unknown,NaN,True


### Clean filtered table

We clean column names and text values again to make sure the filtered table has a consistent format before splitting variables into groups.

In [15]:
filtered.columns = (
    filtered.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

text_cols = ["variable", "category", "tunable", "simulink_usage", "scenario_relevance"]

for col in text_cols:
    if col in filtered.columns:
        filtered[col] = filtered[col].fillna("").astype(str).str.strip()

filtered.head()

,variable,category,tunable,explanation,equation,flagged,simulink_usage,simulink_users,scenario_relevance,reason_filtering,keep_for_current_pipeline
0,weight,fixed_patient_target,No,Patient body weight,NaN,NaN,not_directly_used,NaN,active_or_unknown,NaN,True
1,BSA,fixed_patient_target,No,Body surface area: estimated total surface are...,NaN,NaN,not_directly_used,NaN,active_or_unknown,NaN,True
2,hr,fixed_patient_target,No,Heart rate,NaN,NaN,directly_used,MyComplexModel_V13R2023a/Constant10 | MyComple...,active_or_unknown,NaN,True
3,SAP,fixed_patient_target,No,Systolic arterial pressure; origin: Gupta,NaN,NaN,not_directly_used,NaN,active_or_unknown,NaN,True
4,DAP,fixed_patient_target,No,Diastolic arterial pressure; origin: Gupta,NaN,NaN,not_directly_used,NaN,active_or_unknown,NaN,True


### Basic validation

Before creating the groups, we check that the required columns exist and that variable names are unique.

In [16]:
required_cols = [
    "variable",
    "category",
    "tunable",
    "simulink_usage",
    "scenario_relevance",
    "keep_for_current_pipeline",
]

missing_cols = [col for col in required_cols if col not in filtered.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

duplicates = filtered[filtered["variable"].duplicated(keep=False)]

if len(duplicates) > 0:
    display(duplicates[["variable", "category", "tunable"]])
    raise ValueError("Duplicate variable names found in filtered table.")

print("Filtered table looks valid.")
print("Number of variables:", len(filtered))

Filtered table looks valid.
Number of variables: 90


### Split variables by category

Variables are separated according to their classification. These groups will be reused by the patient-specific setup, sampling, simulation, and ML notebooks.

In [17]:
fixed_patient_targets = filtered[
    filtered["category"] == "fixed_patient_target"
]["variable"].tolist()

derived_physiological_variables = filtered[
    filtered["category"] == "derived_physiological_variable"
]["variable"].tolist()

derived_model_parameters = filtered[
    filtered["category"] == "derived_model_parameter"
]["variable"].tolist()

fixed_model_constants = filtered[
    filtered["category"] == "fixed_model_constant"
]["variable"].tolist()

uncertain_coefficients = filtered[
    filtered["category"] == "uncertain_coefficient"
]["variable"].tolist()

controller_parameters = filtered[
    filtered["category"] == "controller_parameter"
]["variable"].tolist()

derived_controller_parameters = filtered[
    filtered["category"] == "derived_controller_parameter"
]["variable"].tolist()

pump_parameters = filtered[
    filtered["category"].str.contains("pump_parameter", na=False)
]["variable"].tolist()

helper_variables = filtered[
    filtered["category"] == "helper_variable"
]["variable"].tolist()

model_switches = filtered[
    filtered["category"] == "model_switch"
]["variable"].tolist()

simulation_output_targets = filtered[
    filtered["category"] == "simulation_output_target"
]["variable"].tolist()

simulation_signals = filtered[
    filtered["category"] == "simulation_signal"
]["variable"].tolist()

### Split tunable coefficients

The uncertain coefficients are divided into first-stage tunables and later-stage tunables. Only first-stage tunables will be sampled in the first calibration experiment.

In [18]:
stage_1_tunables = filtered[
    (filtered["category"] == "uncertain_coefficient") &
    (filtered["tunable"].str.lower() == "yes")
]["variable"].tolist()

later_tunables = filtered[
    (filtered["category"] == "uncertain_coefficient") &
    (filtered["tunable"].str.lower().isin(["yes later", "maybe later"]))
]["variable"].tolist()

print("Stage 1 tunables:")
for var in stage_1_tunables:
    print("-", var)

print("\nLater tunables:")
for var in later_tunables:
    print("-", var)

Stage 1 tunables:
- k_Vtot
- k_Vsys
- k_Vusv_sys
- k_Vusv_sys_ven
- k_Vusv_pulm_ven
- k_Ctot
- k_Csys
- k_Rsysven
- k_Rpulmart
- k_ESP_LV
- k_ESP_RV

Later tunables:


## Output target selection

Not every simulation output needs to be used in the calibration loss. Primary output targets are used for calibration, while the remaining outputs are kept as diagnostics.

In [19]:
primary_output_targets = [
    "LAP_real",
    "RAP_real",
    "SAP_real",
    "DAP_real",
    "sPAP_real",
    "dPAP_real",
    "EDV_LV_real",
    "ESV_LV_real",
    "EDV_RV_real",
    "ESV_RV_real",
    "CO_real",
]

# Keep only outputs that actually exist in the filtered table
primary_output_targets = [
    var for var in primary_output_targets
    if var in simulation_output_targets
]

diagnostic_outputs = [
    var for var in simulation_output_targets
    if var not in primary_output_targets
]

print("Primary output targets:")
for var in primary_output_targets:
    print("-", var)

print("\nDiagnostic outputs:")
for var in diagnostic_outputs:
    print("-", var)

Primary output targets:
- LAP_real
- RAP_real
- SAP_real
- DAP_real
- sPAP_real
- dPAP_real
- EDV_LV_real
- ESV_LV_real
- EDV_RV_real
- ESV_RV_real
- CO_real

Diagnostic outputs:
- EDV_LA_real
- ESV_LA_real
- EF_LA_real
- MAP_real
- EDP_LV_out
- ESP_LV_out
- SV_LV_real
- EF_LV_real
- EDV_RA_real
- ESV_RA_real
- EF_RA_real
- EDP_RV_out
- ESP_RV_out
- SV_RV_real
- EF_RV_real
- PAP_real


## Target-to-output mapping

This mapping connects fixed patient target values to their corresponding simulated output values. It will later be used to calculate the calibration loss.

In [20]:
target_output_mapping = {
    "LAP": "LAP_real",
    "CVP": "RAP_real",
    "SAP": "SAP_real",
    "DAP": "DAP_real",
    "sPAP": "sPAP_real",
    "dPAP": "dPAP_real",
    "EDV_LV": "EDV_LV_real",
    "ESV_LV": "ESV_LV_real",
    "EDV_RV": "EDV_RV_real",
    "ESV_RV": "ESV_RV_real",
    "CO": "CO_real",
}

# Keep only mappings where both variables exist
target_output_mapping = {
    target: output
    for target, output in target_output_mapping.items()
    if (target in fixed_patient_targets or target in derived_physiological_variables)
    and output in simulation_output_targets
}
# Add derived CO target if CO_real is used as a primary output target
if "CO_real" in primary_output_targets:
    target_output_mapping["CO"] = "CO_real"

derived_calibration_targets = ["CO"]

target_output_mapping

{'LAP': 'LAP_real',
 'CVP': 'RAP_real',
 'SAP': 'SAP_real',
 'DAP': 'DAP_real',
 'sPAP': 'sPAP_real',
 'dPAP': 'dPAP_real',
 'EDV_LV': 'EDV_LV_real',
 'ESV_LV': 'ESV_LV_real',
 'EDV_RV': 'EDV_RV_real',
 'ESV_RV': 'ESV_RV_real',
 'CO': 'CO_real'}

### Flagged variables

Variables with warnings or check-required notes are stored separately so they can be reviewed before running large simulations.

In [21]:
if "flagged" in filtered.columns:
    flagged_variables = filtered[
        filtered["flagged"].notna() &
        (filtered["flagged"].astype(str).str.strip() != "")
    ][["variable", "category", "flagged"]].copy()
else:
    flagged_variables = pd.DataFrame(columns=["variable", "category", "flagged"])

flagged_variables

,variable,category,flagged
68,EDP_LV_out,simulation_output_target,Check required. Code uses EDP_LV_out = max(LVP...
69,ESP_LV_out,simulation_output_target,Check required. Code uses ESP_LV_out = min(LVP...
78,EDP_RV_out,simulation_output_target,Check required. Same issue as LV: code uses ma...
79,ESP_RV_out,simulation_output_target,Check required. Same issue as LV: code uses mi...


All flagged variables have been cleared and are valid 

### Create reusable variable groups

All variable groups are stored in one dictionary. This makes it easy to reload the same configuration in later notebooks.

In [22]:
variable_groups = {
    "scenario": "pediatric_DCM_noVAD_linear",
    "fixed_patient_targets": fixed_patient_targets,
    "derived_calibration_targets": derived_calibration_targets,
    "derived_physiological_variables": derived_physiological_variables,
    "derived_model_parameters": derived_model_parameters,
    "fixed_model_constants": fixed_model_constants,
    "uncertain_coefficients": uncertain_coefficients,
    "stage_1_tunables": stage_1_tunables,
    "later_tunables": later_tunables,
    "controller_parameters": controller_parameters,
    "derived_controller_parameters": derived_controller_parameters,
    "pump_parameters": pump_parameters,
    "helper_variables": helper_variables,
    "model_switches": model_switches,
    "simulation_output_targets": simulation_output_targets,
    "primary_output_targets": primary_output_targets,
    "diagnostic_outputs": diagnostic_outputs,
    "simulation_signals": simulation_signals,
    "target_output_mapping": target_output_mapping,
}
variable_groups

{'scenario': 'pediatric_DCM_noVAD_linear',
 'fixed_patient_targets': ['weight',
  'BSA',
  'hr',
  'SAP',
  'DAP',
  'CVP',
  'LAP',
  'sPAP',
  'dPAP',
  'EDV_LV',
  'ESV_LV',
  'EDV_RV',
  'ESV_RV'],
 'derived_calibration_targets': ['CO'],
 'derived_physiological_variables': ['Csysart', 'Cpulmart'],
 'derived_model_parameters': ['Csysven',
  'Cpulmven',
  'Rsysven',
  'Rsysart',
  'Rpulmart',
  'Rpulmven',
  'Vusv_sys_ven',
  'Vusv_pulm_ven',
  'Vs_LV',
  'ESP_LV',
  'Ps_LV',
  'Vs_RV',
  'ESP_RV',
  'Ps_RV',
  'Lmv',
  'Ltv',
  'Lav',
  'Lpv'],
 'fixed_model_constants': ['V0_LV',
  'Vd_LV',
  'Poffset_LV',
  'V0_RV',
  'Vd_RV',
  'Poffset_RV',
  'Rdir_MV',
  'Rinv_MV',
  'Rdir_TV',
  'Rinv_TV',
  'Rdir_AV',
  'Rinv_AV',
  'Rdir_PV',
  'Rinv_PV'],
 'uncertain_coefficients': ['k_Vtot',
  'k_Vsys',
  'k_Vusv_sys',
  'k_Vusv_sys_ven',
  'k_Vusv_pulm_ven',
  'k_Ctot',
  'k_Csys',
  'k_Rsysven',
  'k_Rpulmart',
  'k_ESP_LV',
  'k_ESP_RV'],
 'stage_1_tunables': ['k_Vtot',
  'k_Vsys',
  'k_

### Save grouped variables

The grouped variables are saved as JSON and CSV files. The JSON file is useful for Python notebooks, while the CSV files are easy to inspect manually.

In [23]:
output_dir = processed_dir / "variable_groups"
output_dir.mkdir(parents=True, exist_ok=True)

# Save JSON
json_path = output_dir / "variable_groups_pediatric_noVAD_linear.json"

with open(json_path, "w") as f:
    json.dump(variable_groups, f, indent=4)

# Save individual CSV files
pd.DataFrame({"variable": fixed_patient_targets}).to_csv(
    output_dir / "fixed_patient_targets_pediatric_noVAD_linear.csv",
    index=False
)

pd.DataFrame({"variable": stage_1_tunables}).to_csv(
    output_dir / "stage_1_tunables_pediatric_noVAD_linear.csv",
    index=False
)

pd.DataFrame({"variable": later_tunables}).to_csv(
    output_dir / "later_tunables_pediatric_noVAD_linear.csv",
    index=False
)

pd.DataFrame({"variable": simulation_output_targets}).to_csv(
    output_dir / "simulation_output_targets_pediatric_noVAD_linear.csv",
    index=False
)

pd.DataFrame({"variable": primary_output_targets}).to_csv(
    output_dir / "primary_output_targets_pediatric_noVAD_linear.csv",
    index=False
)

pd.DataFrame(
    list(target_output_mapping.items()),
    columns=["target_variable", "simulation_output"]
).to_csv(
    output_dir / "target_output_mapping_pediatric_noVAD_linear.csv",
    index=False
)

flagged_variables.to_csv(
    output_dir / "flagged_variables_pediatric_noVAD_linear.csv",
    index=False
)

print("Saved variable groups to:")
print(json_path)

Saved variable groups to:
..\01_Data\processed_pediatric_noVAD_linear\variable_groups\variable_groups_pediatric_noVAD_linear.json


### Summary

This notebook created reusable variable groups for the pediatric DCM no-VAD linear scenario. The filtered variables were separated into patient targets, first-stage tunables, later tunables, model parameters, simulation outputs, and diagnostic outputs. These files will be used in the next notebook for patient-specific target values and tunable parameter bounds.

In [24]:
summary = {
    "Total filtered variables": len(filtered),
    "Fixed patient targets": len(fixed_patient_targets),
    "Stage 1 tunables": len(stage_1_tunables),
    "Later tunables": len(later_tunables),
    "Simulation output targets": len(simulation_output_targets),
    "Primary output targets": len(primary_output_targets),
    "Diagnostic outputs": len(diagnostic_outputs),
    "Flagged variables": len(flagged_variables),
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["Group", "Count"]
)

summary_df

,Group,Count
0,Total filtered variables,90
1,Fixed patient targets,13
2,Stage 1 tunables,11
3,Later tunables,0
4,Simulation output targets,27
5,Primary output targets,11
6,Diagnostic outputs,16
7,Flagged variables,4
